# Stage 4: Exploratory Data Analysis (EDA)
## Project: AI-Based Student Dropout Prediction and Early Warning System

**Objective:** Understand student dropout patterns, academic performance variations, and socio-economic risk factors through descriptive statistics and visualizations.

> **Important Scope Note:**
> - We analyze **only** the training dataset (`train_data.csv`) to prevent data leakage into the test set.
> - All correlations observed are **statistical associations**, not causal proofs.
> - Nominal categorical features (e.g., Course, Nationality) are analyzed separately from continuous numerical variables.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Load training set and metadata
train_path = os.path.join("..", "dataset", "train_data.csv")
meta_path = os.path.join("..", "models", "feature_metadata.json")

df = pd.read_csv(train_path)
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print(f"Training set loaded successfully: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

### 1. Target Class Distribution (`Dropout_Risk`)
- `0`: Retained (Graduate or Enrolled)
- `1`: At-Risk (Dropout)

In [ ]:
counts = df['Dropout_Risk'].value_counts()
pcts = df['Dropout_Risk'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Students': counts, 'Percentage (%)': pcts.round(2)})
display(dist_df)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(['Retained (0)', 'At-Risk (1)'], counts, color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
axes[0].set_title('Student Count by Risk Class')
axes[0].set_ylabel('Number of Students')

axes[1].pie(counts, labels=['Retained (0)', 'At-Risk (1)'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], explode=(0, 0.08))
axes[1].set_title('Target Class Proportion')
plt.show()

### 2. Academic Performance Comparison: 1st Semester
Comparing average grades and approved curricular units between retained and at-risk students.

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x='Dropout_Risk', y='Curricular units 1st sem (grade)', hue='Dropout_Risk', legend=False, palette=['#2ecc71', '#e74c3c'])
plt.xticks([0, 1], ['Retained (0)', 'At-Risk (1)'])
plt.title('1st Semester Grades: Retained vs. At-Risk')
plt.ylabel('Grade (0 to 20 scale)')
plt.show()

plt.figure(figsize=(8, 4.5))
sns.kdeplot(data=df, x='Curricular units 1st sem (approved)', hue='Dropout_Risk', palette=['#2ecc71', '#e74c3c'], common_norm=False, fill=True)
plt.title('1st Semester Approved Units Density')
plt.show()

### 3. Socio-Economic and Financial Health Indicators
How tuition fee payment, debtor status, and scholarships associate with dropout rates.

In [ ]:
for factor, name in [
    ('Tuition fees up to date', 'Tuition Fees Status'),
    ('Debtor', 'Debtor Status'),
    ('Scholarship holder', 'Scholarship Holder Status')
]:
    rate = df.groupby(factor)['Dropout_Risk'].mean() * 100
    print(f"--- {name} ---")
    print(rate.round(2).to_string())
    print()

### 4. Correlation Heatmap for Continuous Numerical Features
We strictly isolate continuous numerical metrics and exclude integer-coded nominals (like Course, Nationality).

In [ ]:
num_cols = meta.get('continuous_numerical_features', []) + ['Dropout_Risk']
corr = df[num_cols].corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f', annot_kws={'size': 8})
plt.title('Correlation Heatmap of Continuous Numerical Features')
plt.show()